# 05 · Col check: normalisation, units, restart, scheduling

**Run first:**
```bash
uv run pytest -v tests/unit/common/test_hashing.py tests/unit/common/test_stats.py tests/unit/common/test_dates.py
uv run pytest -v tests/unit/dtrack_left/checks/test_col_check.py tests/unit/dtrack_right/checks/test_col_check.py
```
**Read:** `common/hashing.py`, `dtrack_left/checks/col_check.py::ColCheckPipeline.run`,
`dtrack_right/checks/col_check.py::DrainScheduler`.

In [ ]:
import nbkit
import pandas as pd
from nbkit import table

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## 1 · `collapse_decimal` touches pure decimals only

Trailing zeros go, leading zeros and exponents stay. A narrow rule leaves fewer ways for the
two sides to disagree; `007` may be a real business key.

In [ ]:
from dtrack_right.common.hashing import collapse_decimal, normalize_value

values = ["7.50", "7.500", "7.00", "2.0", "-0.10", "007", "1e3", "7", "abc"]
table(
    pd.DataFrame(
        {
            "raw": values,
            "collapse_decimal": [collapse_decimal(v) for v in values],
            "numeric key": [normalize_value(v, "numeric") for v in values],
            "string key (from upper case)": [normalize_value(v.upper(), "string") for v in values],
        }
    )
)

## 2 · Units = windows × columns, summed over pairs

In [ ]:
folder = nbkit.sandbox("05")
data, store = nbkit.stage_one(folder)
rows = []
for key, manifest in store.list_json("manifest/"):
    if key.endswith((".keys.json", "_CLOSED")):
        continue
    rows.append(
        {
            "pair": manifest["pair"],
            "windows": len(manifest["windows"]),
            "columns": len(manifest["columns"]),
            "expected_units": manifest["expected_units"],
        }
    )
frame = pd.DataFrame(rows)
table(frame)
print("total units:", frame.expected_units.sum())

## 3 · A restarted `col-check` only adds what is missing

The pipeline reads `_INDEX/` first, skips every `unit_id` already there, and continues the
sequence after the highest number, so lexical order still equals commit order.

In [ ]:
from dtrack_left.checks.col_check import (
    ColCheckPipeline,
    LeftRuntime,
    publish_fixed_keys,
    wait_for_manifests,
)

manifests = wait_for_manifests(store, timeout=1, poll_interval=0)
for manifest in manifests:
    publish_fixed_keys(store, manifest, folder / "left_work")
runtime = LeftRuntime(folder / "left_work", store)
pipeline = lambda: ColCheckPipeline(
    store, pull_window=runtime.pull_window, compute_stats=runtime.compute_stats
)
print("first run:", pipeline().run(manifests))
for key in store.list_keys("_INDEX/")[5:]:
    (store.root / key).unlink()
print("pointers left after a simulated crash:", len(store.list_keys("_INDEX/")))
print("second run:", pipeline().run(manifests))
print(store.list_keys("_INDEX/")[4:7])

## 4 · `--wip`: back-pressure when nobody consumes

`ready_unclaimed()` counts units published but not yet compared. Above `--wip` the Left side
sleeps and re-checks, so it never races far ahead of a Right job that is slow or not started.

In [ ]:
print("published but unclaimed:", store.ready_unclaimed())

## 5 · The Right scheduler: comparisons first, prefetch when idle

A = a unit whose Right statistics are ready → compare now, the result is final.
B = compute the Right statistics of a window before its units arrive, so workers never idle.

In [ ]:
from dtrack_right.checks.col_check import DrainScheduler

scheduler = DrainScheduler(prefetch_depth=2)
scheduler.add_manifest(
    {"pair": "orders", "windows": [{"bucket": "2026-01-01"}, {"bucket": "2026-02-01"}]}
)
scheduler.add_unit({"pair": "orders", "window": "2026-02-01", "column": "AMOUNT"})
first = scheduler.next_task()
print("1:", first.kind, first.window, "(the window that already has a waiting unit goes first)")
scheduler.mark_right_ready("orders", "2026-02-01", {"amount": {}})
second = scheduler.next_task()
print("2:", second.kind, second.window, second.payload["column"])
third = scheduler.next_task()
print("3:", third.kind, third.window, "(speculative)")